# Módulo 4 — Estrutura de Projetos com Claude Code

Organizar bem um projeto de dados com Claude Code aumenta muito a eficácia das interações. Neste notebook exploramos o CLAUDE.md, skills e boas práticas de organização.

## O que vamos ver

1. Estrutura de projeto recomendada
2. Como escrever um CLAUDE.md eficaz
3. Skills — tarefas reutilizáveis para Claude
4. Boas práticas de versionamento
5. Integração com Jupyter

---

## 1. Estrutura de Projeto Recomendada

Uma boa estrutura para projetos de dados com Claude Code:

```
projeto-qualidade-dados/
├── CLAUDE.md                    ← Contexto do projeto para o Claude
├── .env                         ← Credenciais (nunca versionar!)
├── .gitignore                   ← Ignorar .env, __pycache__, etc.
├── requirements.txt
├── .claude/
│   └── commands/                ← Skills (slash commands)
│       ├── data-quality.md
│       ├── generate-report.md
│       └── validate-schema.md
├── src/
│   ├── __init__.py
│   ├── conexao_snowflake.py     ← Módulo de conexão
│   ├── validacoes.py            ← Funções de validação
│   └── relatorios.py            ← Geração de relatórios
├── notebooks/
│   ├── 01_exploracao.ipynb
│   └── 02_qualidade_mensal.ipynb
├── dados/
│   ├── amostras/                ← CSVs pequenos para testes
│   └── relatorios/              ← Relatórios gerados
└── docs/
    └── dicionario_dados.md
```

In [ ]:
import os

# Verificar estrutura de um projeto
def listar_estrutura(caminho: str, nivel: int = 0, max_nivel: int = 3) -> None:
    """Lista a estrutura de diretórios de forma hierárquica."""
    if nivel > max_nivel:
        return
    
    try:
        itens = sorted(os.listdir(caminho))
    except PermissionError:
        return
    
    # Ignorar itens ocultos e __pycache__
    ignorar = {".git", "__pycache__", ".ipynb_checkpoints", ".venv", "node_modules"}
    itens = [i for i in itens if i not in ignorar]
    
    for i, item in enumerate(itens):
        eh_ultimo = i == len(itens) - 1
        prefixo = "    " * nivel + ("└── " if eh_ultimo else "├── ")
        item_path = os.path.join(caminho, item)
        
        if os.path.isdir(item_path):
            print(f"{prefixo}{item}/")
            listar_estrutura(item_path, nivel + 1, max_nivel)
        else:
            tamanho = os.path.getsize(item_path)
            print(f"{prefixo}{item} ({tamanho:,} bytes)")

# Listar estrutura do curso
raiz = os.path.dirname(os.path.dirname(os.path.abspath(".")))
print("Estrutura do projeto curso:")
# Mostrar estrutura atual do diretório
listar_estrutura("../", max_nivel=2)

## 2. Anatomia de um CLAUDE.md Eficaz

O CLAUDE.md é carregado pelo Claude Code em toda sessão. Veja o exemplo em `exemplos/CLAUDE.md`.

In [ ]:
# Lendo o CLAUDE.md de exemplo
with open("exemplos/CLAUDE.md", "r", encoding="utf-8") as f:
    conteudo_claude_md = f.read()

print(conteudo_claude_md)

In [ ]:
# Checklist para um bom CLAUDE.md
checklist = [
    ("Contexto do negócio", "Qual é o domínio, quem são os usuários do sistema?"),
    ("Stack tecnológica", "Linguagem, bibliotecas, banco de dados e versões exatas"),
    ("Estrutura do projeto", "Onde estão os principais módulos e arquivos?"),
    ("Convenções de código", "snake_case, prefixos, estilo de imports"),
    ("Convenções de SQL", "Schemas, prefixos de tabelas/views, estilo de queries"),
    ("O que NUNCA fazer", "DELETE sem WHERE, commitar .env, etc."),
    ("Comandos úteis", "Como rodar testes, gerar relatórios, conectar ao banco"),
    ("Dados sensíveis", "Quais campos são PII, como devem ser tratados"),
]

print("Checklist CLAUDE.md — seu arquivo contém:")
print()
for item, descricao in checklist:
    presente = item.lower().replace(" ", "") in conteudo_claude_md.lower().replace(" ", "")
    status = "[OK]" if presente else "[ ] Adicionar!"
    print(f"  {status} {item}")
    if not presente:
        print(f"         → {descricao}")

## 3. Skills — Tarefas Reutilizáveis

In [ ]:
# Lendo o arquivo de skill de exemplo
with open("exemplos/exemplo_skill.md", "r", encoding="utf-8") as f:
    conteudo_skill = f.read()

print("=== Skill: data-quality-check ===")
print(conteudo_skill)

## 4. Boas Práticas de Versionamento

Com Claude Code, seu `.gitignore` precisa de atenção especial.

In [ ]:
gitignore_recomendado = """
# Credenciais — NUNCA versionar
.env
.env.*
*.key
credentials.json

# Python
__pycache__/
*.py[cod]
.venv/
*.egg-info/

# Jupyter
.ipynb_checkpoints/

# Dados — não versionar dados reais de produção
dados/producao/
dados/relatorios/*.xlsx
*.csv
!dados/amostras/*.csv  # exceto amostras de teste

# Outputs de notebooks
*.png
*.pdf

# IDE
.vscode/
.idea/
"""

print(".gitignore recomendado para projetos de dados com Claude Code:")
print(gitignore_recomendado)

In [ ]:
# Boas práticas de commit quando usando IA
exemplos_commit = [
    {
        "mensagem": "feat: adicionar validação de CPF/CNPJ no cadastro de UC",
        "descricao": "Implementa verificação de formato e dígitos verificadores. Gerado com assistência do Claude Code.",
        "boa_pratica": True,
        "motivo": "Descreve o que foi feito e menciona o uso de IA"
    },
    {
        "mensagem": "update",
        "descricao": "",
        "boa_pratica": False,
        "motivo": "Não descreve nada — impossível rastrear o que mudou"
    },
    {
        "mensagem": "fix: corrigir cálculo de DEC quando não há interrupções no mês",
        "descricao": "Tratamento de ZeroDivisionError quando total_ucs = 0. Bug encontrado via testes.",
        "boa_pratica": True,
        "motivo": "Descreve o bug corrigido e como foi encontrado"
    },
]

for ex in exemplos_commit:
    status = "BOA PRÁTICA" if ex["boa_pratica"] else "EVITAR"
    print(f"\n[{status}]")
    print(f"  Mensagem: {ex['mensagem']}")
    if ex['descricao']:
        print(f"  Descrição: {ex['descricao']}")
    print(f"  Motivo: {ex['motivo']}")